# Practice 4. Finding the right clause: diagnose and fix

**ML in Industry 2026 · Topic 4: Retrieval-augmented generation**

## Goal

An engineer asks: *"After a bearing change, pump P-101 vibrates at 4.8 mm/s. Is this normal?"* The answer is in
regulation R-12, but the plant archive now has 16 documents, with old revisions, near-copies and a manual in English.
You measure where answers get lost, fix one cause at a time, and compare each fix with the step before it on the
same questions.

**Glossary: measuring**

- **chunk**: a piece of a document (clause, table, table row) that search returns.
- **retrieval set**: 143 questions with known answers and known source chunks.
- **slice**: a group of questions of one kind, e.g. "the question names a past date".
- **gold evidence**: the chunks that hold the answer, found from the document structure.
- **recall@5**: the chunk with the answer is among the first five search results.
- **evidence recall**: every needed chunk (value, table, action) reached the model.
- **end-to-end**: the final answer is right: the value with its unit, or the right status.
- **correct + verified**: right, and every citation passes the three checks of step 7.
- **false refusal**: "not in the documents" or "needs a person" although the documents answer.
- **invented value**: a value given where the documents have none.
- **token**: a piece of text a model reads or writes; it measures size and cost.
- **ablation table**: one change per row; each row is compared with the row above.
- **paired comparison**: both rows answer the same questions; we count questions fixed and broken.
- **fixed / broke**: questions the new row fixed, and questions it broke.
- **verdict**: "B is better": new row B beats row A, and not by chance.
- **p**: the chance to see such a difference by luck; below 0.05 we call it a real difference.
- **[a%–b%]**: the true share lies in this range with 95% confidence.

**Glossary: the rows of the table**

- **tag**: the code of a piece of equipment, e.g. pump P-101.
- **S0**: plain BM25 keyword search: chunks ranked by the words they share with the question.
- **S1–S6, the ladder**: six plant steps: tags, table rows, applicability, date filter, evidence pack, contract.
- **S2b**: table rows searched apart from the clauses, at most two rows per question.
- **applicability**: the equipment registry tells which documents apply to this piece of equipment.
- **evidence pack**: the found chunks plus the clauses they refer to and their sections.
- **answer contract**: a fixed answer form with a status, a value and its sources.
- **statuses**: answered; not in the documents; conflict (documents disagree); needs a person.
- **vectors**: numbers an embedding model computes for the meaning of a text.
- **M1**: dense search, by meaning, not by words.
- **M2**: hybrid search, keywords and dense search together.
- **M3, reranker**: the course language model puts the top 20 chunks in order.
- **M4, cross-encoder**: a small reranking model that scores each chunk against the question.
- **M1t**: an English question is translated into Russian before search.
- **@S0 / @S6**: the mechanism instead of the plant steps / on top of them.
- **replay**: saved model answers; the same request gets the same answer offline.
- **holdout**: 27 extra questions the fixes were never tuned on.

## Steps

| Step | You should see |
|------|----------------|
| 0 Final table | all rows at once |
| 1 Setup | the model answers, or replay |
| 2 Archive | 16 documents, 440 chunks |
| 3 Questions | 143 in 12 slices |
| 4 S0 | half right, losses in search |
| 5 S1–S5 | each step fixes its slice |
| 6 Mechanisms | wording fixed, dates not |
| 7 Contract | checked citations, conflicts |
| 8 Audit, holdout | a record, a gate, new questions |
| 9 Check | your numbers equal the reference |

**Live or replay.** Search runs on your laptop; model answers come from replay. `Mode: REPLAY` with a course
token: check `.env` (README, "Common problems").

## 0. Where we are going

The final table of this practice. Each row adds one change to the row in "compared with" and is compared with it
on the same questions.

In [1]:
import warnings

import pandas as pd
from IPython.display import Markdown, display

from plant_assistant import REFERENCE_DIR
from plant_assistant.display import show

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
reference = pd.read_csv(REFERENCE_DIR / "ablation.csv").fillna("")
show(reference, ["config", "vs", "recall@5", "evidence recall", "end-to-end", "e2e verdict"])

,row,compared with,recall@5,evidence recall,end-to-end,verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],no detectable difference (p = 0.062)
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],no detectable difference (p = 1.000)
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],no detectable difference (p = 0.625)
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],B is better (p = 0.000)
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],B is better (p = 0.016)
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],B is better (p = 0.000)
7,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],129/143 = 90% [84%–94%],no detectable difference (p = 0.070)
8,M1t,S0,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],B is better (p = 0.021)
9,M1@S0,S0,84/123 = 68% [60%–76%],76/123 = 62% [53%–70%],90/143 = 63% [55%–70%],B is better (p = 0.019)


## 1. Setup

Model settings are in `models.toml`, the URL and the key in `.env`. The answers use the course model Qwen 3.8 27B
with thinking off: no hidden reasoning before the answer. Dense search and the cross-encoder use saved vectors and
scores; with access to those models you can rebuild them (see README).

In [2]:
import os

from plant_assistant import REPLAY_DIR, RUNS_DIR, STORE_DIR
from plant_assistant.config import get_profile, list_profiles, load_env
from plant_assistant.llm import CallLogger, LLMClient, ReplayClient

load_env()
PROFILE = os.environ.get("PRACTICE_PROFILE", "course")
pd.DataFrame(list_profiles().items(), columns=["profile", "description"])

,profile,description
0,course,"Course model Qwen 3.8 27B via LiteLLM + vLLM, thinking off"
1,course-thinking,"Course model Qwen 3.8 27B, thinking on (spends tokens before the JSON: raise max_tokens)"
2,local,"Small open-weight model on your laptop via Ollama (Qwen3.5 4B, Q4)"
3,embed-bge-m3,Multilingual embeddings bge-m3 (567M) via Ollama: ollama pull bge-m3
4,embed-qwen3,Multilingual embeddings Qwen3-Embedding 0.6B via Ollama: ollama pull qwen3-embedding:0.6b
5,embed-course,"e5-mistral-7b-instruct (4096-dim) on the course server: questions with an instruction, documents without"
6,rerank-course,"Cross-encoder bge-reranker-v2-m3 on the course server: POST <server root>/rerank, one call per question"
7,drafter,Model that drafts holdout questions: gpt-oss-120b on the course server


In [3]:
def model_answers(profile_name: str) -> bool:
    # One call without network retries: if the model is not there, we want to know now.
    try:
        client = LLMClient(get_profile(profile_name), max_network_retries=0)
    except ValueError as exc:  # profile not configured (no URL or key)
        print(f"{profile_name}: {exc}")
        return False
    r = client.chat([{"role": "user", "content": "Reply with one word: OK"}])
    print(f"{profile_name}: {r.content.strip()[:20]!r} ({r.latency_s:.1f}s)" if r.ok else r.error[:200])
    return r.ok


LIVE = os.environ.get("PRACTICE_MODE") != "replay" and model_answers(PROFILE)
replay = {task: ReplayClient(REPLAY_DIR / f"{task}_course.jsonl")
          for task in ("answers", "answers_thinking", "rerank", "translate", "crossencoder")}
live = LLMClient(get_profile(PROFILE), logger=CallLogger(RUNS_DIR / f"my-{PROFILE}" / "calls.jsonl")) if LIVE \
    else replay["answers"]
print("Mode:", f"LIVE single questions on {PROFILE}, the full set from replay" if LIVE else
      "REPLAY of the lecturer's saved answers (course model)")
from plant_assistant.search import VectorStore

vectors = sorted((REPLAY_DIR / "vectors").glob("*.npz"))
print("Embedding vectors:", [v.stem for v in vectors] or "none: dense rows (M1, M2) are not measured")
VECTORS = VectorStore(vectors[0]) if vectors else None

Mode: REPLAY of the lecturer's saved answers (course model)
Embedding vectors: ['e5-mistral']


## 2. The plant archive, version 2

The four documents of practice 3 plus 12 new synthetic ones. All were created as digital files, not scans, so
reading the PDFs does not get in the way. The scale is small (tens of pages, 440 chunks); the process is the same at any scale. The
document registry (`data/documents.csv`) gives each document its dates of validity and the stations it covers.

In [4]:
from plant_assistant.corpus import build_corpus, load_registry
from plant_assistant.store import load_store

summary = build_corpus(STORE_DIR)
print(summary)
_, chunks = load_store(STORE_DIR)
by_id = {c.chunk_id: c for c in chunks}
docs = pd.DataFrame(load_registry())
docs[["doc_id", "revision", "amendment", "kind", "language", "effective_from", "effective_to", "scope_stations",
      "scope_equipment_types", "scope_tags", "pages", "source"]]

{'documents': 16, 'pages': 47, 'chunks': 440, 'chunks_without_rows': 330, 'table_rows': 110, 'index_version': 'corpus-5719c6dc76.parser-a53367.emb-none', 'seconds': 5.3}


,doc_id,revision,amendment,kind,language,effective_from,effective_to,scope_stations,scope_equipment_types,scope_tags,pages,source
0,DS-P-205,NaN,NaN,datasheet,zh,2023-01-01,None,[2],[pump],[P-205],1,practice 3
1,PS-101,NaN,NaN,datasheet,ru,2019-01-01,None,[2],[pump],[P-101],1,practice 3
2,R-12,2.0,NaN,regulation,ru,2022-06-01,2025-03-14,[2],"[pump, compressor, fan]",[],6,practice 3
3,R-12,3.0,NaN,regulation,ru,2025-03-14,None,[2],"[pump, compressor, fan]",[],6,practice 3
4,FS-07,2.0,NaN,instruction,ru,2024-01-15,None,[2],[all],[],3,practice 4
5,LAB-03,1.0,NaN,procedure,ru,2022-03-01,None,[lab],[all],[],2,practice 4
6,OM-P-205,2.0,NaN,manual,en,2023-05-15,None,[2],[pump],[P-205],3,practice 4
7,PS-102,NaN,NaN,datasheet,ru,2018-01-01,None,[2],[pump],[P-102],1,practice 4
8,PS-310,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-310],1,practice 4
9,PS-311,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-311],1,practice 4


**What is new.** An **amendment**, an official change to one clause, moves a check interval of R-12 from 720 h
to 500 h on 01.09.2026. R-14 copies R-12 for pump station 3, with stricter limits. The English manual of pump
P-205 gives milder limits than R-12. The valve list VL-2 has a table of 110 rows over four pages.

In [5]:
from plant_assistant.display import where

for cid in ("R-12-r3-table1", "R-14-r1-table1"):
    c = by_id[cid]
    print(f"{where(c)}  scope_stations={c.scope_stations}  valid {c.effective_from} → {c.effective_to or 'now'}")
    print("   ", c.text.splitlines()[0], "|", [line for line in c.text.splitlines() if "|II|" in line][0])

R-12 rev. 3 Table 3  scope_stations=['2']  valid 2025-03-14 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|4,5|7,1|
R-14 rev. 1 Table 3  scope_stations=['3']  valid 2024-03-01 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|3,5|5,6|


Each chunk now also knows when it is valid, which stations and equipment it covers, and which clauses it refers
to and is referred to by.

In [6]:
fields = ["chunk_id", "clause", "table_no", "row_key", "effective_from", "effective_to", "amended_by", "references",
          "referenced_by"]
pd.DataFrame([{f: getattr(by_id[cid], f) for f in fields} for cid in
              ("R-12-r3-4.4.1", "R-12-r3-a1-4.4.1", "R-12-r3-4.3.3", "R-12-r3-6.2", "R-20-r2-3.3",
               "VL-2-r4-tА.2-V-12")])

,chunk_id,clause,table_no,row_key,effective_from,effective_to,amended_by,references,referenced_by
0,R-12-r3-4.4.1,4.4.1,None,None,2025-03-14,2026-09-01,R-12-r3-a1-4.4.1,[],[R-12-r3-a1-4.4.1]
1,R-12-r3-a1-4.4.1,4.4.1,None,None,2026-09-01,None,None,[R-12-r3-4.4.1],[]
2,R-12-r3-4.3.3,4.3.3,None,None,2025-03-14,None,None,[R-12-r3-table1],[R-12-r3-6.2]
3,R-12-r3-6.2,6.2,None,None,2025-03-14,None,None,"[R-12-r3-4.3.3, R-12-r3-4.3.4]",[R-20-r2-3.3]
4,R-20-r2-3.3,3.3,None,None,2023-01-10,None,None,"[R-12-r2-6.2, R-12-r3-6.2, R-14-r1-6.2]",[]
5,VL-2-r4-tА.2-V-12,А,А.2,V-12,2025-11-20,None,None,[],[]


The **index version** is a fingerprint of the chunks, the PDF reader and the embedding model. Every answer record
keeps it; when it changes, the questions run again.

In [7]:
print("index_version:", summary["index_version"])
pd.crosstab(pd.Series([c.doc_id for c in chunks], name="document"), pd.Series([c.chunk_type for c in chunks],
                                                                              name="chunk type"), margins=True)

index_version: corpus-5719c6dc76.parser-a53367.emb-none


chunk type,clause,datasheet,note,section,table,table_row,All
document,,,,,,,
DS-P-205,0,1,0,0,0,0,1
FS-07,16,0,0,5,1,0,22
LAB-03,12,0,0,4,0,0,16
OM-P-205,21,0,0,7,3,0,31
PS-101,0,1,0,0,0,0,1
PS-102,0,1,0,0,0,0,1
PS-310,0,1,0,0,0,0,1
PS-311,0,1,0,0,0,0,1
R-12,86,0,2,28,6,0,122


## 3. The retrieval set

143 questions in 12 slices. The 50 questions of practice 3 form the base slice; 20 of them also go to search in
English. Questions are asked in Russian, as at a Russian plant; the English translation is for reading. Gold
evidence is found by code from the document structure. Slice results are counts: 3 of 4, not 75%.

In [8]:
from plant_assistant.cases import counts_by_slice, load_cases

cases = load_cases()
case_by_id = {c.case_id: c for c in cases}
show(pd.DataFrame([counts_by_slice(cases)], index=["questions"]))

,base (practice 3),English wording,applicability,past date,incident,what changed,several chunks,two documents,conflict,table row,tag spelling,not in documents
questions,50,20,12,10,3,4,8,3,9,8,6,10


In [9]:
from plant_assistant.display import gold_frame

P101 = case_by_id["m01"]
print("Question (asked in Russian):", P101.question_ru)
print("Translation:                ", P101.question_en)
print("Expected:", P101.expected, "| asked on", P101.asked_on)
gold_frame(P101)

Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
Translation:                 After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
Expected: {'values': ['4,5', '24'], 'unit': 'h'} | asked on 2026-10-27


,role,locator (any of),gold chunk ids
0,answer,R-12 r3 cl 4.3.3,R-12-r3-4.3.3
1,support,R-12 r3 t 3,R-12-r3-table1
2,action,R-12 r3 cl 6.1,R-12-r3-6.1
3,action,R-12 r3 cl 6.2,R-12-r3-6.2


One answer needs four chunks: the clause with the deadline, Table 3 with the limit, and two clauses with the
actions. recall@5 looks only for the chunk with the value; evidence recall needs all four.

## 4. Baseline S0: keyword search, top 5, no filters

Search runs live; the model answers for all 143 questions come from replay.

In [10]:
from plant_assistant import reval
from plant_assistant.display import answer_frame, context_frame, hits_frame
from plant_assistant.pipeline import CONFIGS, LADDER, System

system = System(chunks, answer_client=replay["answers"], rerank_client=replay["rerank"],
                translate_client=replay["translate"], cross_client=replay["crossencoder"], vectors=VECTORS,
                think_client=replay["answers_thinking"])
records, scores = {}, {}


def run(name: str) -> pd.DataFrame:
    records[name] = system.run(cases, CONFIGS[name], progress=False)
    scores[name] = reval.score(cases, records[name], by_id)
    return scores[name]


run("S0")
show(reval.ablation({"S0": scores["S0"]}, {}), ["config", "recall@5", "evidence recall", "end-to-end",
                                                 "correct + verified", "abstain ok", "conflict ok"])

,row,recall@5,evidence recall,end-to-end,correct + verified,no answer: refused,conflicts shown
0,S0,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],47/143 = 33% [26%–41%],10/10,0/9


In [11]:
display(Markdown("**Wrong answers of S0 by link**"))
pd.DataFrame([reval.losses(scores["S0"])], index=["S0"])

**Wrong answers of S0 by link**

,reading,search,assembly,model
S0,0,60,2,8


A wrong answer is charged to the first link that lost it: reading the PDF, search, building the context, or the
model. Most answers are lost **in search**: the chunk with the answer never reached the model. Nothing is lost in
reading, because the documents were created as digital files, not scans. Below: the top five search results for the P-101 question.

In [12]:
rec = next(r for r in records["S0"] if r["case_id"] == "m01")
hits_frame(rec, P101, by_id)

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 cl. 4.4.2,2022-06-01 → 2025-03-14,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
2,R-12 rev. 3 cl. 4.4.2,2025-03-14 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
3,R-14 rev. 1 cl. 4.4.2,2024-03-01 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
4,R-15 rev. 1 cl. 4.3,2023-09-01 → now,,4.3 Марку смазочного материала берут из паспорта насоса. Замена марки без согласования с главным механиком не
5,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.


None of the five holds the answer. They share "после замены подшипника" (after a bearing change) with the
question: one clause in three regulations, and two maintenance clauses. Table 3 and the inspection deadline are
missing. **Task 1 of the lecture** used five failures of this run:

In [13]:
display(Markdown((REFERENCE_DIR / "s0_failures.md").read_text(encoding="utf-8")))

# Five failed answers of S0 (task 1 of lecture 4)

Configuration S0: BM25 over raw chunks, top-5, no filters, the course model answers from the top-5.
Task: for each answer write where it was lost: R = reading, S = search, C = context assembly, M = model.

## 1. m01 (multi_evidence) — lost in: **search**

- Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
- Translation: After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
- Expected: ['4,5', '24'] h
- System answer: The sources do not specify the vibration limit to determine if 4.8 mm/s is normal, but they mandate that measurements be taken within 8 hours after startup following a bearing replacement. You must perform the measurement within this 8-hour window. (value: None)
- Gold evidence: R-12 r3 cl 4.3.3; R-12 r3 t 3; R-12 r3 cl 6.1; R-12 r3 cl 6.2
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; R-15 rev. 1 cl. 4.3; R-15 rev. 1 cl. 5.1
- Why this link: no chunk with the answer reached the context

## 2. g01 (tag_variant) — lost in: **search**

- Question (asked in Russian): Какая мощность двигателя у насоса Р-101?
- Translation: What is the motor power of pump Р-101 (Cyrillic Р)?
- Expected: 55 kW
- System answer: The provided sources do not contain the specific power rating of the motor for pump R-101. (value: None)
- Gold evidence: PS-101
- Top-5 of search: R-12 rev. 2 cl. 4.5.1; R-12 rev. 3 cl. 4.5.1; R-14 rev. 1 cl. 4.5.1; R-15 rev. 1 cl. 5.1; R-12 rev. 2 cl. 7.2
- Why this link: no chunk with the answer reached the context

## 3. v03 (point_in_time) — lost in: **search**

- Question (asked in Russian): С какой периодичностью сейчас нужно контролировать вибрацию насоса P-101 переносным виброметром?
- Translation: How often must the vibration of pump P-101 be checked with a portable meter now?
- Expected: 500 h
- System answer: Vibration of pump P-101 must be controlled with a portable vibrometer every 1000 operating hours, but not less than once a month. (value: 1000)
- Gold evidence: R-12 r3a1 cl 4.4.1
- Top-5 of search: R-12 rev. 2 cl. 3.7; R-12 rev. 3 cl. 3.7; R-14 rev. 1 cl. 3.7; R-12 rev. 2 cl. 4.2.1; R-12 rev. 2 cl. 4.4.1
- Why this link: no chunk with the answer reached the context

## 4. m06 (multi_evidence) — lost in: **assembly**

- Question (asked in Russian): После замены подшипника насос P-101 запустили. В течение скольких часов нужно измерить вибрацию и с какой границей сравнить результат?
- Translation: Pump P-101 was started after a bearing change. Within how many hours must the vibration be measured, and against which limit?
- Expected: ['8', '4,5'] h
- System answer: Vibration measurements must be taken within 8 hours after startup. The sources do not specify the comparison limit. (value: 8)
- Gold evidence: R-12 r3 cl 4.4.2; R-12 r3 t 3
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; FS-07 rev. 2 cl. 3.3; R-12 rev. 2 cl. 6.1
- Why this link: the chunk with the answer is in the context, a required support or action chunk is not

## 5. i01 (incident) — lost in: **model**

- Question (asked in Russian): 10.02.2025 насос P-101 вибрировал 4,8 мм/с, осмотр в течение суток не провели. Было ли это нарушением регламента?
- Translation: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?
- Expected: ['5,0'] mm/s
- System answer: Yes, it was a violation of the regulation. The vibration of 4.8 mm/s exceeded the inspection threshold of 4.5 mm/s for a Class II pump, requiring an inspection within 24 hours, which was not performed. (value: 4,5)
- Gold evidence: R-12 r2 t 3; R-12 r2 cl 4.3.3
- Top-5 of search: R-12 rev. 2 cl. 4.3.3; R-12 rev. 3 cl. 4.3.3; R-14 rev. 1 cl. 4.3.3; R-12 rev. 2 Table 3; R-12 rev. 3 Table 3
- Why this link: every needed chunk was in the context, the answer is still wrong


## 5. The ladder: one plant step per row

Each step is added to the one before. We look at one question before and after, then at the table.

**S1: one spelling of tags.** "Р-101" with a Cyrillic Р, "P101" and "P 102" become P-101 and P-102, in the
chunks and in the question.

In [14]:
for name in ("S0", "S1"):
    rec = system.run_case(case_by_id["g02"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['g02'].question_ru}"), hits_frame(rec, case_by_id["g02"], by_id))

**S0** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,"4.1 Подшипники качения насосов смазывают каждые 3000 часов наработки смазкой, указанной в паспорте насоса."
2,R-14 rev. 1 Table А.1,2024-03-01 → now,,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 cl. 2.2,2024-01-15 → now,,2.2 Проходы к насосам и электрическим щитам шириной не менее 1 м содержат свободными.
4,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.
5,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"


**S1** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 3 Table А.1,2025-03-14 → now,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 Table 1,2024-01-15 → now,,Таблица 1 — Нормы оснащения помещения насосной станции № 2 |Средство|Количество|Место| |---|---|---| |Огнетуши
4,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,"4.1 Подшипники качения насосов смазывают каждые 3000 часов наработки смазкой, указанной в паспорте насоса."
5,PS-101,2019-01-01 → now,answer,# ПАСПОРТ НАСОСА # PUMP DATASHEET <table> <tr> <th>Тег / Tag</th> <td>P-101</td> </tr> <tr> <th>Изго


**S2: table rows as chunks.** Before, a page of the valve table (33 rows) is one chunk. Now each row is a chunk
that repeats the table title, the column names and the units:

In [15]:
print(by_id["VL-2-r4-tА.2-V-12"].text)
for name in ("S1", "S2"):
    rec = system.run_case(case_by_id["r01"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['r01'].question_ru}"), hits_frame(rec, case_by_id["r01"], by_id, 3))

Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; PN, МПа: 1,6; Среда: нефть; Трубопровод: нагнетание P-101; Привод: ручной


**S1** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.
3,R-15 rev. 1 cl. 6.2,2023-09-01 → now,,6.2 При превышении допустимой утечки начальник смены создаёт заявку с приоритетом «средний» по R-20.


**S2** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,VL-2 rev. 4 Table А.2 row V-12,2025-11-20 → now,answer,"Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; P"
3,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.


**S2b: rows searched apart.** Mixed with the clauses, 110 short rows push the datasheets out of the results for
any question that names a pump. So rows get their own search and at most two places. The questions that mixed
rows broke:

In [16]:
from plant_assistant.pipeline import PREVIOUS

previous = dict(PREVIOUS)
broken = ["d01", "d02", "d04", "z01", "z02", "z04", "z05", "i02", "g01"]
for name in ("S1", "S2", "S2b"):
    run(name)
pd.DataFrame({name: scores[name].set_index("case_id").loc[broken, "correct"] for name in ("S1", "S2", "S2b")}).T

case_id,d01,d02,d04,z01,z02,z04,z05,i02,g01
S1,False,False,True,True,True,True,True,True,True
S2,False,False,False,False,False,False,False,False,False
S2b,False,False,True,True,True,True,True,True,True


**S3: applicability.** The equipment registry gives each pump its station, class and documents. Search then sees
only the documents for that station. Pump P-310 is at station 3, so R-14 applies, not R-12. The manual of P-205
never names the pump; only the registry links them.

In [17]:
from plant_assistant.applicability import load_registry

print(load_registry()["P-205"].describe())
rec = system.run_case(case_by_id["f01"], CONFIGS["S3"], with_answer=False)
print("linked by the registry:", [where(by_id[c]) for c in rec["linked"]])

P-205: pump, Pump station 2, motor 37 kW, machine class II, documents DS-P-205, OM-P-205
linked by the registry: ['DS-P-205', 'OM-P-205 rev. 2 Table 1', 'OM-P-205 rev. 2 Table 2', 'OM-P-205 rev. 2 Table 3']


Two pumps of station 3, before and after S3: P-310 at 4.0 mm/s (`p01`) and P-311 at 3.0 mm/s (`p07`). Without
the registry, the second one is judged by R-12.

In [18]:
from plant_assistant.applicability import equipment_context

# single questions: live if the course model answers (the reranker too), replay otherwise
system_live = System(chunks, answer_client=live, rerank_client=live if LIVE else replay["rerank"],
                     translate_client=live if LIVE else replay["translate"], cross_client=replay["crossencoder"],
                     vectors=VECTORS, think_client=replay["answers_thinking"])
for cid in ("p01", "p07"):  # P-310 at 4.0 mm/s; P-311 (class I, station 3) at 3.0 mm/s
    case = case_by_id[cid]
    for name in ("S2", "S3"):
        rec = system_live.run_case(case, CONFIGS[name])
        a = rec["answer"] or {}
        display(Markdown(f"**{cid} · {name}** · {case.question_ru} · scope: {rec['scope'] or 'none'}"),
                hits_frame(rec, case, by_id, 3))
        print(f"value {a.get('value')} (expected {case.expected['values'][0]}), right {reval.grade(case, a)[0]}:",
              a.get("answer"))

**p01 · S2** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: none

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 2 Table 3,2022-06-01 → 2025-03-14,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-12 rev. 3 Table 3,2025-03-14 → now,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"


value 3,5 (expected 3,5), right True: Yes, pump P-310 requires inspection because its vibration of 4.0 mm/s exceeds the inspection threshold of 3.5 mm/s for Class II pumps. The pump should be inspected immediately.


**p01 · S3** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: stations=['3'], types=['pump'], tags=['P-310']

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-14 rev. 1 Table 3,2024-03-01 → now,answer,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-14 rev. 1 cl. 3.3,2024-03-01 → now,,"3.3 Граница осмотра: значение виброскорости, при превышении которого оборудование осматривают в установленный"


value 3,5 (expected 3,5), right True: Yes, the pump P-310 must be inspected because its vibration of 4.0 mm/s exceeds the inspection limit of 3.5 mm/s for a Class II machine. The inspection must be performed within 24 hours, and measurements must be repeated every shift until the inspection is complete.


... (2 outputs left out)

value 4,5 (expected 3,5), right False: No, pump P-311 does not need to be stopped because its vibration of 3.0 mm/s is below the stop limit of 4.5 mm/s for Class I pumps, though it exceeds the inspection limit of 2.8 mm/s.


**p07 · S3** · Насос P-311 вибрирует 3,0 мм/с. Его нужно останавливать? · scope: stations=['3'], types=['pump'], tags=['P-311']

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-14 rev. 1 Table 3,2024-03-01 → now,answer,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-14 rev. 1 cl. 3.4,2024-03-01 → now,,"3.4 Граница останова: значение виброскорости, при превышении которого оборудование немедленно останавливают."


value 3,5 (expected 3,5), right True: No, pump P-311 does not need to be stopped because its vibration velocity of 3.0 mm/s is below the stop limit of 3.5 mm/s for machine class I pumps. It exceeds the inspection limit of 2.2 mm/s, so an inspection is required, but immediate shutdown is not mandated.


**S4: a date filter.** Search keeps only chunks valid on the date of the question. The same question about the
check interval on three dates: 1000 h, 720 h, then 500 h after the amendment. At S4 the amendment still misses
the top five; the evidence pack of S5 brings it.

In [19]:
for cid in ("v01", "v02", "v03"):
    for name in ("S3", "S4", "S5"):
        rec = system_live.run_case(case_by_id[cid], CONFIGS[name])
        a = rec["answer"] or {}
        print(f"{cid} {name}: as_of {rec['as_of']}  expected {case_by_id[cid].expected['value']} h  "
              f"answer {a.get('value')} {a.get('unit')}  top-1 {where(by_id[rec['retrieved'][0]])}")

v01 S3: as_of 2024-05-01  expected 1000 h  answer 720 hours  top-1 R-12 rev. 2 section 4.4
v01 S4: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 section 4.4
v01 S5: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 section 4.4
v02 S3: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 2 section 4.4
v02 S4: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 section 4.4
v02 S5: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 section 4.4
v03 S3: as_of 2026-10-27  expected 500 h  answer 1000 hours  top-1 R-12 rev. 2 cl. 3.7
v03 S4: as_of 2026-10-27  expected 500 h  answer None None  top-1 R-12 rev. 3 cl. 3.7
v03 S5: as_of 2026-10-27  expected 500 h  answer 500 hours  top-1 R-12 rev. 3 cl. 3.7


**Two wrong ways to filter by date.** Filtering after search can leave nothing. A flag "in force now" answers a
question about an incident in February 2025 with today's revision.

In [20]:
from plant_assistant.filters import make_filter, post_filter, status_in_force
from plant_assistant.query import extract_rules

q = case_by_id["i01"]
fields = extract_rules(q.query, q.asked_on)
idx = system.bm25(CONFIGS["S4"])
by_date = make_filter(fields, None, use_scope=False, use_date=True)
pre = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, by_date)]
post = [where(by_id[h.chunk_id]) for h in post_filter(idx.search(q.query, 5), by_id, by_date)]
flag = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, status_in_force)]
print("question:", q.question_en)
pd.DataFrame({"pre-filter by date": pd.Series(pre), "post-filter by date": pd.Series(post),
              "status = in force": pd.Series(flag)})

question: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?


,pre-filter by date,post-filter by date,status = in force
0,R-12 rev. 2 cl. 4.3.3,R-12 rev. 2 cl. 4.3.3,R-12 rev. 3 cl. 4.3.3
1,R-14 rev. 1 cl. 4.3.3,R-14 rev. 1 cl. 4.3.3,R-14 rev. 1 cl. 4.3.3
2,R-12 rev. 2 Table 3,R-12 rev. 2 Table 3,R-12 rev. 3 Table 3
3,R-14 rev. 1 Table 3,NaN,R-14 rev. 1 Table 3
4,R-12 rev. 2 cl. 6.1,NaN,R-12 rev. 3 cl. 6.1


**S5: the evidence pack.** The found chunks, plus what they refer to, what refers to them, and their sections, up
to 2500 tokens. The question: after a bearing change, within how many hours to measure, and against which limit
(`m06`). At S4 the deadline clause is first but Table 3 is missing; the pack adds it:

In [21]:
m06 = case_by_id["m06"]
for name in ("S4", "S5"):
    rec = system.run_case(m06, CONFIGS[name], with_answer=False)
    ctx = context_frame(rec, m06, by_id)
    print(f"{name}: top-1 {where(by_id[rec['retrieved'][0]])}; {len(ctx)} chunks, {ctx.tokens.sum()} tokens; "
          f"gold in the context: {sorted(set(ctx.gold) - {''})}; "
          f"evidence recall = {reval.retrieval_metrics(m06, rec)['evidence_recall']}")
ctx[ctx.gold != ""]

S4: top-1 R-12 rev. 3 cl. 4.4.2; 6 chunks, 688 tokens; gold in the context: ['answer']; evidence recall = False
S5: top-1 R-12 rev. 3 cl. 4.4.2; 36 chunks, 1941 tokens; gold in the context: ['answer', 'support']; evidence recall = True


,chunk,gold,tokens
0,R-12 rev. 3 cl. 4.4.2,answer,34
5,R-12 rev. 3 Table 3,support,115


The ladder in the table, each row against the row above:

In [22]:
for name in ("S3", "S4", "S5"):
    run(name)
cols = ["config", "vs", "recall@5", "evidence recall", "end-to-end", "e2e: fixed / broke", "e2e verdict"]
show(reval.ablation(scores, previous), cols)

,row,compared with,recall@5,evidence recall,end-to-end,fixed / broke,verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],+5 / -0,no detectable difference (p = 0.062)
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],+6 / -7,no detectable difference (p = 1.000)
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],+3 / -1,no detectable difference (p = 0.625)
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],+24 / -2,B is better (p = 0.000)
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],+7 / -0,B is better (p = 0.016)
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],+14 / -0,B is better (p = 0.000)


In [23]:
display(Markdown("**End-to-end by slice** (counts)"))
show(reval.by_slice(scores, "correct"))

**End-to-end by slice** (counts)

,base (practice 3),English wording,applicability,past date,incident,what changed,several chunks,two documents,conflict,table row,tag spelling,not in documents
config,,,,,,,,,,,,
S0,41/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/9,6/8,0/6,10/10
S1,41/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/9,6/8,5/6,10/10
S2,38/50,5/20,4/12,4/10,0/3,1/4,3/8,0/3,0/9,8/8,4/6,10/10
S2b,41/50,5/20,1/12,5/10,1/3,1/4,3/8,0/3,0/9,8/8,5/6,10/10
S3,49/50,10/20,8/12,6/10,1/3,1/4,3/8,0/3,0/9,8/8,6/6,10/10
S4,49/50,10/20,10/12,9/10,3/3,1/4,3/8,0/3,0/9,8/8,6/6,10/10
S5,50/50,11/20,12/12,10/10,3/3,3/4,7/8,3/3,0/9,8/8,6/6,10/10


- S1 fixes tag spellings; S2 fixes table rows but breaks base questions; S2b keeps the fix without the damage.
- S3 fixes applicability, S4 questions about a past date, S5 questions that need several chunks.
- Conflicts stay at zero until S6: a simple answer cannot say "the documents disagree".

## 6. Mechanisms: rows of the same table

Search methods from the Advanced NLP course, added in two places: instead of the plant steps (@S0) or on top of
them (@S6). M1 searches by meaning, M2 mixes keywords and meaning, M3 and M4 put the top 20 chunks in order, M1t
translates an English question first. We do not explain how they work; we look at where they help.

In [24]:
MECH = ["S0", "M1@S0", "M2@S0", "M3@S0", "M4@S0", "S6", "M1@S6", "M2@S6", "M3@S6", "M4@S6"]
for name in ["M1t"] + MECH:
    if name not in scores and (VECTORS is not None or CONFIGS[name].retriever == "bm25"):
        run(name)
table = reval.ablation(scores, previous)
show(table[table.config.isin(MECH + ["M1t"])], cols)

,row,compared with,recall@5,evidence recall,end-to-end,fixed / broke,verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,
7,M1t,S0,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],+9 / -1,B is better (p = 0.021)
8,M1@S0,S0,84/123 = 68% [60%–76%],76/123 = 62% [53%–70%],90/143 = 63% [55%–70%],+32 / -15,B is better (p = 0.019)
9,M2@S0,S0,77/123 = 63% [54%–71%],65/123 = 53% [44%–61%],86/143 = 60% [52%–68%],+23 / -10,B is better (p = 0.035)
10,M3@S0,S0,93/123 = 76% [67%–82%],90/123 = 73% [65%–80%],101/143 = 71% [63%–77%],+32 / -4,B is better (p = 0.000)
11,M4@S0,S0,90/123 = 73% [65%–80%],80/123 = 65% [56%–73%],94/143 = 66% [58%–73%],+27 / -6,B is better (p = 0.000)
12,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],129/143 = 90% [84%–94%],+7 / -1,no detectable difference (p = 0.070)
13,M1@S6,S6,100/123 = 81% [74%–87%],115/123 = 93% [88%–97%],138/143 = 97% [92%–98%],+13 / -4,B is better (p = 0.049)
14,M2@S6,S6,99/123 = 80% [73%–87%],113/123 = 92% [86%–96%],133/143 = 93% [88%–96%],+5 / -1,no detectable difference (p = 0.219)
15,M3@S6,S6,104/123 = 85% [77%–90%],115/123 = 93% [88%–97%],133/143 = 93% [88%–96%],+5 / -1,no detectable difference (p = 0.219)


In [25]:
focus = ["language", "point_in_time", "applicability", "conflict", "table_row", "tag_variant", "multi_evidence"]
show(reval.by_slice({k: scores[k] for k in MECH if k in scores}, "correct")[focus])

,English wording,past date,applicability,conflict,table row,tag spelling,several chunks
config,,,,,,,
S0,5/20,4/10,2/12,0/9,6/8,0/6,3/8
M1@S0,17/20,3/10,3/12,0/9,8/8,6/6,2/8
M2@S0,8/20,7/10,3/12,0/9,8/8,3/6,2/8
M3@S0,10/20,7/10,8/12,0/9,8/8,2/6,3/8
M4@S0,10/20,5/10,6/12,0/9,8/8,2/6,3/8
S6,11/20,10/10,12/12,7/9,8/8,6/6,7/8
M1@S6,20/20,10/10,10/12,9/9,7/8,5/6,8/8
M2@S6,14/20,10/10,12/12,7/9,8/8,6/6,8/8
M3@S6,13/20,10/10,12/12,8/9,8/8,6/6,8/8


**What each row costs.** Seconds and model calls per question, and tokens of the chat model. The language-model
reranker (M3) adds a chat call of about 1800 tokens; the cross-encoder (M4) adds one short call and no chat tokens.

In [26]:
show(reval.cost_table(records).reset_index(), ["config", "median s / question", "model calls / question",
                                                "chat calls / question", "prompt tokens / question",
                                                "completion tokens / question", "cross-encoder tokens / question"])

,row,seconds per question,model calls,chat calls,tokens in,tokens out,cross-encoder tokens
0,S0,1.46,1.00,1.00,843.0,84.0,0.0
1,S1,1.46,1.00,1.00,864.0,84.0,0.0
2,S2,1.40,1.00,1.00,727.0,84.0,0.0
3,S2b,1.49,1.00,1.00,839.0,106.0,0.0
4,S3,1.68,1.00,1.00,1083.0,84.0,0.0
5,S4,1.76,1.00,1.00,1135.0,83.0,0.0
6,S5,3.05,1.00,1.00,2811.0,88.0,0.0
7,M1t,1.93,2.00,2.00,769.0,102.0,0.0
8,M1@S0,1.97,2.00,1.00,1298.0,85.0,0.0
9,M2@S0,1.91,2.00,1.00,1073.0,84.0,0.0


In [27]:
lang = scores["M1t"].case_id
display(Markdown("**English questions**: as they are (S0), translated into Russian (M1t), "
                 "dense search on the English wording (M1@S0)"))
pd.DataFrame({name: [f"{int(scores[name].set_index('case_id').loc[lang, m].astype(bool).sum())}/{len(lang)}"
                     for m in ("recall@k", "correct")] for name in ("S0", "M1t", "M1@S0") if name in scores},
             index=["recall@5", "end-to-end"])

**English questions**: as they are (S0), translated into Russian (M1t), dense search on the English wording (M1@S0)

,S0,M1t,M1@S0
recall@5,3/20,9/20,15/20
end-to-end,5/20,13/20,17/20


- Mechanisms fix wording and language: dense search answers 17 of 20 English questions.
- Only the plant steps fix dates, applicability and conflicts: dense search alone answers 3 of 10 past-date questions.
- The cross-encoder gives most of the reranker's gain in a quarter of the time: 0.5 s extra against 2 s.

## 7. The answer contract and the citation checks

S6 adds a fixed answer form: a status, the value with its unit, and up to five short claims with their sources. The
model may cite only chunks it was given; code then adds the document, revision, clause and page.

**A place to think.** With thinking off, the model writes the answer at once. So the form starts with a short
internal field, `check`: do the sources agree, is the number in the cited text? It is logged for the audit, never
shown to the engineer and never graded.

**Three checks of every citation**: it **exists** (the chunk was given to the model), it is **in force** on the
date of the question, and it **supports** the answer (the number and the unit are in its text). One citation must
support the value; the others back other claims and are checked in part 2 by a judge (a grading model).

**The P-101 question.** At S6, keyword search puts Table 3 sixth, so the evidence pack has nothing to start from.
The reranker on top (M3@S6) brings Table 3 back. At S6 the answer is a refusal: it states no number, so it passes
the checks; "verified" does not mean "right".

In [28]:
for name in ("S6", "M3@S6"):
    rec = system_live.run_case(P101, CONFIGS[name])
    print(f"{name}: top-5 {[where(by_id[c]) for c in rec['top']]}")
    print(f"     status {rec['answer']['status']}, correct {reval.grade(P101, rec['answer'])[0]}, "
          f"verified {rec['checks']['verified']}")
answer, checks = answer_frame(rec)
display(answer)
checks.insert(1, "where", checks.chunk_id.map(lambda c: where(by_id[c]) if c else ""))  # R-12-r3-table3 is Table А.1
checks

S6: top-5 ['R-12 rev. 3 cl. 4.4.2', 'R-15 rev. 1 cl. 4.3', 'R-15 rev. 1 cl. 5.1', 'R-15 rev. 1 cl. 4.1', 'FS-07 rev. 2 Table 1']
     status not_in_documents, correct False, verified True
M3@S6: top-5 ['R-12 rev. 3 Table А.1', 'R-12 rev. 3 Table 3', 'R-12 rev. 3 section 6', 'R-20 rev. 2 Table 1', 'R-12 rev. 3 cl. 4.4.2']
     status answered, correct True, verified True


,value
field,
status,answered
answer,"No, 4.8 mm/s exceeds the inspection limit of 4.5 mm/s for Class II pumps. You must inspect the equipment within 24 h..."
value,"4,5"
unit,мм/с


,source,where,chunk_id,exists,in_force,supports
0,R-12-r3-table3,R-12 rev. 3 Table А.1,R-12-r3-table3,True,True,False
1,R-12-r3-table1,R-12 rev. 3 Table 3,R-12-r3-table1,True,True,True
2,R-12-r3-4.3.3,R-12 rev. 3 cl. 4.3.3,R-12-r3-4.3.3,True,True,True
3,R-12-r3-6.2,R-12 rev. 3 cl. 6.2,R-12-r3-6.2,True,True,False
4,R-20-r2-table1,R-20 rev. 2 Table 1,R-20-r2-table1,True,True,False


**Task 3 of the lecture**: "Inspect P-101 within 24 hours (R-12, rev. 3, clause 4.3.4)". The clause exists and is
in force, but it is about the stop limit; "24 hours" is in the next clause, 4.3.3. The model can cite only chunks
it was given, but it can still cite the wrong one. The checks catch it.

In [29]:
from plant_assistant.answer import Answer, Claim
from plant_assistant.assemble import render
from plant_assistant.verify import check_answer

context = ["R-12-r3-4.3.4", "R-12-r3-4.3.3", "R-12-r2-4.3.3"]  # rev. 2 is there, as for a "what changed" question
_, sources = render(context, by_id, show_dates=True, ids_as_labels=True)
rows = []
for cid in context + ["R-12-r3-6.2"]:  # the last one was not in the context
    claim = Answer(status="answered", answer="Inspect P-101 within 24 hours.", value="24", unit="h",
                   claims=[Claim(text="Inspect within 24 hours.", sources=[cid])])
    r = check_answer(claim, sources, chunks, by_id, P101.asked_on)
    rows.append({"cited id": cid, "exists (in the context)": r["exists"], "in force": r["in_force"],
                 "supports 24 h": r["supports"], "verified": r["verified"]})
pd.DataFrame(rows)

,cited id,exists (in the context),in force,supports 24 h,verified
0,R-12-r3-4.3.4,True,True,False,False
1,R-12-r3-4.3.3,True,True,True,True
2,R-12-r2-4.3.3,True,False,True,False
3,R-12-r3-6.2,False,True,False,False


**A conflict.** For pump P-205, R-12 says inspect above 4.5 mm/s; the English manual says alarm at 7.1 mm/s. For
lubrication of P-101, R-12 says every 2000 h, the maintenance regulation R-15 every 3000 h. The system must show
both sides, not choose:

In [30]:
for cid in ("f01", "f05"):
    rec = system_live.run_case(case_by_id[cid], CONFIGS["S6"])
    cited = sorted({by_id[c["chunk_id"]].doc_id for c in rec["checks"]["citations"] if c["chunk_id"]})
    print(f"{cid}: {rec['answer']['status']} | cited {cited} | {rec['answer']['answer'][:220]}")

f01: conflict | cited ['OM-P-205', 'R-12'] | Conflicting limits found. Plant regulation R-12 (Table 3) sets the inspection limit for Pump Class II at 4.5 mm/s. Manufacturer manual OM-P-205 (Table 2) sets the alarm limit at 7.1 mm/s. Both documents are in force; cla
f05: conflict | cited ['R-12', 'R-15'] | Conflicting maintenance intervals found. Document R-15 (clause 4.1) specifies lubrication every 3000 hours, while Document R-12 (clause 7.1) specifies every 2000 hours. Both documents are in force; clarification is requi


**Not in the documents.** Valve V-912 is in no document; valve V-112 is only in the registry. Both "not in the
documents" and "needs a person" are right here: no value is given.

In [31]:
for cid in ("x01", "x03"):
    rec = system_live.run_case(case_by_id[cid], CONFIGS["S6"])
    print(cid, rec["answer"]["status"], "|", rec["answer"]["answer"][:160])

x01 not_in_documents | The specific allowable pressure for valve V-912 is not stated in the provided documents. Clause 1.3 of VL-2 (rev. 4) requires operating pressure to not exceed t
x03 not_in_documents | The provided documents do not contain information about the allowable pressure for valve V-112.


**The contract in three versions.** An early version had no length limit: long answers were cut off, and the
model sometimes reasoned inside the answer. The final version adds limits and the `check` field. Thinking mode was
worse and three times slower. S6:v1 is the early version, S6:v2 adds limits, S6 is the final version with `check`,
S6:think is thinking mode. The table counts invalid answers, reasoning found in the answer, and answers handed to
a person.

In [32]:
for name in ("S6:v1", "S6:v2", "S6:think"):
    if name not in scores:
        run(name)
pairs = {"S6:v2": "S6:v1", "S6": "S6:v2", "S6:think": "S6:v2"}
ba = reval.ablation({k: scores[k] for k in ("S6:v1", "S6:v2", "S6", "S6:think")}, pairs)
show(ba, ["config", "vs", "end-to-end", "e2e: fixed / broke", "e2e verdict", "false refusals", "wrong or invented",
          "invalid", "leaks", "fallbacks"])

,row,compared with,end-to-end,fixed / broke,verdict,false refusals,wrong or invented,invalid answers,reasoning in the answer,handed to a person
0,S6:v1,,130/143 = 91% [85%–95%],,,12/123 = 10% (1 needs a person),1,0,0,0
1,S6:v2,S6:v1,128/143 = 90% [83%–94%],+0 / -2,no detectable difference (p = 0.500),12/123 = 10% (1 needs a person),3 (1 invented),0,0,0
2,S6,S6:v2,129/143 = 90% [84%–94%],+2 / -1,no detectable difference (p = 1.000),11/123 = 9%,3,0,0,0
3,S6:think,S6:v2,123/143 = 86% [79%–91%],+1 / -6,no detectable difference (p = 0.125),14/123 = 11% (3 needs a person),6 (1 invented),0,0,1


**What `check` shows.** Pump P-205 at 6.0 mm/s, where R-12 and the manual disagree (`f03`). At S6 the model sees
both and still picks R-12, by a rule nobody gave it. `check` does not fix this, but the log shows why, so a person
can fix the rule or the prompt. With dense search or the reranker on top, the answer is right.

In [33]:
f03 = case_by_id["f03"]
for name in ("S6:v2", "S6"):
    rec = system_live.run_case(f03, CONFIGS[name])
    print(f"{name}: {rec['answer']['status']} | answer: {rec['answer']['answer'][:200]}")
    if rec.get("check"):
        print(f"      check (internal): {rec['check']}")

S6:v2: answered | answer: Yes, 6.0 mm/s is permissible as it is below the 7.1 mm/s alarm limit and the 7.1 mm/s stop limit for a Class II pump. No specific action is required by the sources for this value, but it must be recor
S6: answered | answer: No, 6.0 mm/s is not acceptable for normal operation as it exceeds the inspection limit of 4.5 mm/s for Class II pumps. The shift supervisor must create a high-priority CMMS request for inspection with
      check (internal): P-205 is Class II (R-12 Table A.1). R-12 Table 3 sets limits for Pumps Class II: Inspection 4.5 mm/s, Stop 7.1 mm/s. Measured 6.0 mm/s is > 4.5 and < 7.1. OM-P-205 Table 2 sets Alarm at 7.1 mm/s. R-12 is the plant regulation (in force 2025-03-14) and applies. 6.0 exceeds the inspection limit, so it 


The whole ladder with S6:

In [34]:
full = reval.ablation(scores, previous)
show(full[full.config.isin([c.name for c in LADDER])], cols + ["correct + verified", "abstain ok", "conflict ok"])

,row,compared with,recall@5,evidence recall,end-to-end,fixed / broke,verdict,correct + verified,no answer: refused,conflicts shown
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,,47/143 = 33% [26%–41%],10/10,0/9
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],+5 / -0,no detectable difference (p = 0.062),51/143 = 36% [28%–44%],10/10,0/9
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],+6 / -7,no detectable difference (p = 1.000),52/143 = 36% [29%–45%],10/10,0/9
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],+3 / -1,no detectable difference (p = 0.625),51/143 = 36% [28%–44%],10/10,0/9
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],+24 / -2,B is better (p = 0.000),62/143 = 43% [36%–52%],10/10,0/9
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],+7 / -0,B is better (p = 0.016),109/143 = 76% [69%–82%],10/10,0/9
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],+14 / -0,B is better (p = 0.000),123/143 = 86% [79%–91%],10/10,0/9
12,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],129/143 = 90% [84%–94%],+7 / -1,no detectable difference (p = 0.070),128/143 = 90% [83%–94%],10/10,7/9


**How a row fails matters.** A refusal is safe but costs the engineer time; a wrong value goes into a decision.
S0 fails both ways, S6 almost only by refusing, and M3@S0 gives wrong and invented values.

In [35]:
modes = ["false refusals", "wrong or invented", "right abstentions", "invalid", "fallbacks"]
show(full[full.config.isin(["S0", "S5", "S6", "M3@S0", "M3@S6"])], ["config", "end-to-end"] + modes)

,row,end-to-end,false refusals,wrong or invented,right refusals,invalid answers,handed to a person
0,S0,73/143 = 51% [43%–59%],42/123 = 34%,28 (1 invented),19/20 = 95%,0,0
6,S5,123/143 = 86% [79%–91%],11/123 = 9%,9,20/20 = 100%,0,0
10,M3@S0,101/143 = 71% [63%–77%],15/123 = 12%,27 (3 invented),17/20 = 85%,0,0
12,S6,129/143 = 90% [84%–94%],11/123 = 9%,3,20/20 = 100%,0,0
15,M3@S6,133/143 = 93% [88%–96%],7/123 = 6%,3 (1 invented),19/20 = 95%,0,0


**Needs a person.** Six extra questions where the rule is in the documents but a fact is missing: a pump with no
class in the registry, a tag that matches two pumps, a measurement in one direction of three. The right answer
names what is missing. Giving a value here is the dangerous failure.

In [36]:
from plant_assistant import DATA_DIR

needs_human = load_cases(DATA_DIR / "needs_human.jsonl")
rows = {}
for name in ["S6", "M3@S6", "M4@S6"] + (["M1@S6", "M2@S6"] if VECTORS is not None else []):
    recs = system.run(needs_human, CONFIGS[name], workers=4, progress=False)
    rows[name] = {r["case_id"]: (r.get("answer") or {}).get("status", "invalid") for r in recs}
pd.DataFrame(rows).assign(question=[c.question_en for c in needs_human])

,S6,M3@S6,M4@S6,M1@S6,M2@S6,question
nh01,needs_human,needs_human,needs_human,needs_human,needs_human,Pump P-215 vibrates at 4.0 mm/s. Is that normal?
nh02,needs_human,needs_human,needs_human,needs_human,needs_human,Pump P-220 vibrates at 5.0 mm/s. Must it be stopped?
nh03,not_in_documents,needs_human,needs_human,needs_human,needs_human,A pump at pump station No. 2 vibrates at 4.0 mm/s. Must it be inspected?
nh04,answered,conflict,conflict,needs_human,conflict,Pump P-216 vibrates at 4.0 mm/s after installation. Must it be inspected?
nh05,needs_human,not_in_documents,not_in_documents,needs_human,needs_human,Which grease must be used for the bearings of pump P-215?
nh06,answered,answered,answered,answered,answered,Pump P-101: 4.2 mm/s in the vertical direction; the horizontal and axial directions were not measured. Is the vibrat...


## 8. The audit log and the regression gate

Half a year later, someone asks why the system said 5,0. The **audit record** keeps the question, its date, the
index version, the chunks with their revisions, the answer and the checks.

In [37]:
import json

from plant_assistant.audit import audit_record

rec = next(r for r in records["M3@S6"] if r["case_id"] == "m01")
print(json.dumps(audit_record(rec, system.index_version, by_id), ensure_ascii=False, indent=1)[:1800])

{
 "ts": "2026-10-28T14:52:16+00:00",
 "case_id": "m01",
 "question": "После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?",
 "asked_on": "2026-10-27",
 "as_of": "2026-10-27",
 "config": "M3@S6",
 "index_version": "corpus-5719c6dc76.parser-a53367.emb-none",
 "filters": "stations=['2'], types=['pump'], tags=['P-101']",
 "retrieved": [
  {
   "chunk_id": "R-12-r3-table3",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "А",
   "table_no": "А.1",
   "row_key": null,
   "page": 6,
   "effective_from": "2025-03-14",
   "effective_to": null
  },
  {
   "chunk_id": "R-12-r3-table1",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "4.3",
   "table_no": "3",
   "row_key": null,
   "page": 4,
   "effective_from": "2025-03-14",
   "effective_to": null
  },
  {
   "chunk_id": "R-12-r3-6",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "6",
   "table_no": null,
   "row_key": null

The **regression gate** is a pass/fail test on the question set: the new row must not be clearly worse and may
break at most two questions. Topic 6 runs it after every new revision or index.

In [38]:
g = reval.gate(scores["S5"], scores["S6"])
print(f"S6 vs S5: {g['A right']} -> {g['B right']} of {g['n']}, fixed {g['B fixed']}, broke {g['B broke']} "
      f"{g['broken_cases']}, p = {g['p_value']}  ->  GATE {'PASS' if g['passed'] else 'FAIL'} {g['reasons']}")

S6 vs S5: 123 -> 129 of 143, fixed 7, broke 1 ['w02'], p = 0.0703  ->  GATE PASS []


**Holdout.** 27 questions written by a model from the documents and four job roles only: operator, mechanic,
reliability engineer, auditor. The fixes were never tuned on them, so they show whether the gains carry over.

In [39]:
holdout = load_cases(REPLAY_DIR.parent / "holdout.jsonl")
h_scores = {}
for name in [c.name for c in LADDER] + ["M3@S0", "M3@S6"]:
    h_scores[name] = reval.score(holdout, system.run(holdout, CONFIGS[name], progress=False), by_id)
show(reval.ablation(h_scores, previous), ["config", "vs", "recall@5", "evidence recall", "end-to-end",
                                           "e2e: fixed / broke", "e2e verdict"])

,row,compared with,recall@5,evidence recall,end-to-end,fixed / broke,verdict
0,S0,,13/23 = 57% [37%–74%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],,
1,S1,S0,13/23 = 57% [37%–74%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],+0 / -0,no detectable difference (p = 1.000)
2,S2,S1,15/23 = 65% [45%–81%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],+1 / -1,no detectable difference (p = 1.000)
3,S2b,S1,12/23 = 52% [33%–71%],9/23 = 39% [22%–59%],13/27 = 48% [31%–66%],+0 / -2,no detectable difference (p = 0.500)
4,S3,S2b,15/23 = 65% [45%–81%],12/23 = 52% [33%–71%],20/27 = 74% [55%–87%],+8 / -1,B is better (p = 0.039)
5,S4,S3,17/23 = 74% [54%–87%],17/23 = 74% [54%–87%],19/27 = 70% [52%–84%],+0 / -1,no detectable difference (p = 1.000)
6,S5,S4,17/23 = 74% [54%–87%],18/23 = 78% [58%–90%],20/27 = 74% [55%–87%],+1 / -0,no detectable difference (p = 1.000)
7,S6,S5,17/23 = 74% [54%–87%],18/23 = 78% [58%–90%],21/27 = 78% [59%–89%],+1 / -0,no detectable difference (p = 1.000)
8,M3@S0,S0,20/23 = 87% [68%–95%],20/23 = 87% [68%–95%],23/27 = 85% [68%–94%],+8 / -0,B is better (p = 0.008)
9,M3@S6,S6,19/23 = 83% [63%–93%],20/23 = 87% [68%–95%],24/27 = 89% [72%–96%],+3 / -0,no detectable difference (p = 0.250)


- The plant steps still help on the holdout; applicability (S3) is the big step.
- Between the best rows the differences are within noise: 27 questions are too few.
- Everyday questions rarely ask about dates or conflicts, so a reranker alone does about as well there.

## 9. Check yourself

Your search columns must equal the reference exactly; end-to-end from replay too. A live run may differ by a few
questions.

In [40]:
mine = reval.ablation(scores, previous).set_index("config")
ref = reference.set_index("config")
common = [c for c in mine.index if c in ref.index]
pd.DataFrame({"recall@5 (yours)": mine.loc[common, "recall@5"], "recall@5 (reference)": ref.loc[common, "recall@5"],
              "end-to-end (yours)": mine.loc[common, "end-to-end"],
              "end-to-end (reference)": ref.loc[common, "end-to-end"]})

,recall@5 (yours),recall@5 (reference),end-to-end (yours),end-to-end (reference)
config,,,,
S0,61/123 = 50% [41%–58%],61/123 = 50% [41%–58%],73/143 = 51% [43%–59%],73/143 = 51% [43%–59%]
S1,65/123 = 53% [44%–61%],65/123 = 53% [44%–61%],78/143 = 55% [46%–62%],78/143 = 55% [46%–62%]
S2,69/123 = 56% [47%–65%],69/123 = 56% [47%–65%],77/143 = 54% [46%–62%],77/143 = 54% [46%–62%]
S2b,67/123 = 54% [46%–63%],67/123 = 54% [46%–63%],80/143 = 56% [48%–64%],80/143 = 56% [48%–64%]
S3,82/123 = 67% [58%–74%],82/123 = 67% [58%–74%],102/143 = 71% [63%–78%],102/143 = 71% [63%–78%]
S4,93/123 = 76% [67%–82%],93/123 = 76% [67%–82%],109/143 = 76% [69%–82%],109/143 = 76% [69%–82%]
S5,93/123 = 76% [67%–82%],93/123 = 76% [67%–82%],123/143 = 86% [79%–91%],123/143 = 86% [79%–91%]
M1t,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],13/20 = 65% [43%–82%]
M1@S0,84/123 = 68% [60%–76%],84/123 = 68% [60%–76%],90/143 = 63% [55%–70%],90/143 = 63% [55%–70%]


**Success criteria**

- Your recall@5 and evidence recall equal `reference/ablation.csv`.
- The P-101 question at M3@S6: "not normal, inspect within 24 hours", all three checks passed.
- Valve V-912 gets no value; P-205 and the lubrication of P-101 get "conflict" with both documents cited.

**What you built:** a question set, losses by link, six plant steps, an answer contract with checks, an audit
record and a regression gate.

**Typical mistakes**

- Filtering by date after search: nothing is left.
- "In force now" instead of dates: a past incident gets today's rule.
- A tag normalized in the chunks but not in the question.
- "Dense is better" from two averages, without a paired comparison.
- A citation check that looks only at the clause number.

**Optional tasks** (details in the README)

1. Read the scanned PDF with Docling, the reader of practice 3, and run S6.
2. Another embedding model through Ollama (local models).
3. Query fields extracted by the model instead of by rules.
4. A smaller evidence pack: 500, 1000, 2000 tokens.
5. Near-copies of a regulation without the station fields.
6. A document hierarchy in code: the stricter rule wins.
7. The index in a vector database (pgvector or Qdrant).